# sarcomp smoke test (Colab)

Lightweight check **before** launching the real experiments: are gradients flowing, are there NaNs, do all four
training stages (`ae`, `fm`, `e1d2`, `finetune`) + checkpoints + pipelines + metrics run?

* Uses **synthetic data**, no 3 GB TIFF needed (an optional cell at the bottom checks a real batch).
* Run with a GPU runtime (Runtime > Change runtime type > T4/A100) so the *full-size* check is meaningful.
* Every check prints `PASS` / `FAIL`; the final cell asserts that all passed.

## 1. Get the code
Either clone from GitHub (set `REPO_URL`) or upload/unzip the repo so that the `sarcomp/` folder is in the working directory.

In [ ]:
import os, sys, subprocess
REPO_URL = ""   # e.g. "https://github.com/<user>/<repo>.git"  (leave empty if you uploaded the folder yourself)
if REPO_URL and not os.path.isdir("sarcomp"):
    subprocess.run(["git", "clone", REPO_URL, "repo"], check=True)
    os.chdir("repo")
assert os.path.isdir("sarcomp"), "sarcomp/ not found in %s - clone or upload the repo first" % os.getcwd()
sys.path.insert(0, os.getcwd())
%pip install -q pyyaml

In [ ]:
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 2. Tiny models (seconds)
Checks transform round-trip, complex layers vs. PyTorch's native complex conv, latent shapes, NaN/Inf hooks on every layer, per-part gradient norms, overfit-one-batch, and every training stage through the real `Trainer`.

In [ ]:
from sarcomp.smoke import run_smoke_test
rep_tiny = run_smoke_test(device="auto", size="tiny")

## 3. Full-size models (real widths: 256x256 input, latent1 8x8x1024, latent2 1x1x2048)
Same checks with the real architecture (batch size 2, a couple of training steps). Prints peak GPU memory. Skip on a CPU runtime.

In [ ]:
rep_full = run_smoke_test(device="auto", size="full", steps=2) if torch.cuda.is_available() else None

## 4. Real data from Google Drive (optional but recommended)
Put the ICEYE `.tif` in a Drive folder and set `DRIVE_TIF_DIR` below (edit the path to yours).
The notebook mounts Drive and **copies the TIFF to local Colab disk** first: the loaders do many small random window reads, which are very slow over the Drive mount.
Then it checks one batch per split (finite values, correct shapes) and computes the log-amplitude scale from the train columns.

In [ ]:
import glob, shutil

USE_REAL_DATA = True
DRIVE_TIF_DIR = "/content/drive/MyDrive/complex_data"   # <-- EDIT: Drive folder that contains the ICEYE .tif
LOCAL_DIR = "/content/data"                              # fast local copy used by the loaders

rep_real = None
if USE_REAL_DATA:
    from google.colab import drive
    drive.mount("/content/drive")
    tifs = glob.glob(os.path.join(DRIVE_TIF_DIR, "*.tif"))
    assert tifs, "no .tif found in %s - fix DRIVE_TIF_DIR (folders under MyDrive: %s)" % (
        DRIVE_TIF_DIR, os.listdir("/content/drive/MyDrive")[:20])
    os.makedirs(LOCAL_DIR, exist_ok=True)
    local = os.path.join(LOCAL_DIR, os.path.basename(tifs[0]))
    if not os.path.exists(local):
        print("copying %.1f GB to local disk ..." % (os.path.getsize(tifs[0]) / 1e9))
        shutil.copy(tifs[0], local)
    %pip install -q rasterio
    rep_real = run_smoke_test(device="auto", size="tiny", archs=(), tif_dir=LOCAL_DIR)

## 5. Verdict

In [ ]:
reports = [("tiny", rep_tiny), ("full", rep_full), ("real data", rep_real)]
for name, r in reports:
    if r is not None:
        print(name, "->", r.summary())
assert all(r.ok for _, r in reports if r is not None), "smoke test failed - see FAIL lines above"
print("ALL SMOKE CHECKS PASSED")